# Z3-Python 06 -- Optimisation avancee (twin C#)

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [<< Z3-Python-05 Quantifiers](Z3-Python-05-Quantifiers-Proofs-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Hierarchiser** des contraintes souples avec des poids et des priorites dans un `Optimize`
2. **Gerer** plusieurs objectifs simultanes (maximiser un revenu tout en minimisant un cout)
3. **Enumerer** le front de Pareto pour explorer les compromis entre objectifs contradictoires
4. **Modeliser** des contraintes souples (soft constraints) au moyen de variables de relaxation booleennes (MaxSAT)
5. **Appliquer** ces techniques a un cas pratique d'allocation de budget multi-projets

### Prerequis
- Z3-Python-01-Csharp (Introduction) : `Solver`, `Int`, `Bool`, `Real`, `Optimize` de base
- Z3-Python-03-Csharp (Tactiques) : familiarite avec `Bool`, `Or`, `And`, `MkITE`
- Notions d'optimisation combinatoire (maximisation sous contraintes)

### Duree estimee : ~40 min

---

**Ce notebook** est le twin C# du notebook Python [Z3-Python-06-Advanced-Optimization.ipynb](Z3-Python-06-Advanced-Optimization.ipynb). Il poursuit l'exploration de la classe `Optimize` au-dela du cas elementaire (un seul objectif vu en NB01). Les problemes reels comportent presque toujours **plusieurs objectifs contradictoires** : maximiser la qualite tout en minimisant le cout, ou satisfaire un maximum de preferences quand toutes ne peuvent l'etre simultanement. Z3 fournit pour cela les contraintes ponderees, l'optimisation multi-objectif lexicographique, et l'enumeration du front de Pareto.

> **Kernel** : `.net-csharp` (.NET Interactive). Le moteur est le **vrai** [Microsoft.Z3](https://github.com/Z3Prover/z3) via NuGet `#r "nuget: Microsoft.Z3"` (v4.12.2.0) -- le meme solveur C++ que `z3-solver` Python.

In [1]:
#r "nuget: Microsoft.Z3"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python)
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");
Console.WriteLine("Optimisation multi-criteres : MkOptimize, MkMaximize, MkMinimize, AssertSoft.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Moteur : Microsoft.Z3 Z3 4.12.2.0


Optimisation multi-criteres : MkOptimize, MkMaximize, MkMinimize, AssertSoft.


## 1. Rappel et motivation -- au-dela d'un seul objectif

Le notebook 01 a introduit `Optimize` avec **un** objectif unique : maximiser la valeur d'un sac a dos, ou minimiser le temps de fin d'un ordonnancement. Le patron etait simple :

```csharp
var opt = ctx.MkOptimize();
opt.Assert(contraintes_dures);
opt.MkMaximize(objectif);   // un seul objectif
opt.Check();
```

### La realite est multi-objectif

Dans la vraie vie, les problemes comportent plusieurs objectifs **contradictoires** :

| Domaine | Objectif A (maximiser) | Objectif B (minimiser) |
|---------|----------------------|----------------------|
| Logistique | Qualite de service | Cout de transport |
| Finance | Rendement | Risque |
| Planification | Satisfaction des preferences | Cout horaire |
| Ingenierie | Performance | Consommation energetique |

On ne peut pas « maximiser A et minimiser B » simultanement de facon absolue : il faut choisir une strategie de compromis. Z3 offre trois approches complementaires :

1. **Priorites hierarchiques** (section 2) : chaque contrainte souple a un poids, on minimise la somme ponderee des violations.
2. **Objectifs multiples lexicographiques** (section 3) : on optimise les objectifs les uns apres les autres, par ordre de priorite.
3. **Front de Pareto** (section 4) : on enumere tous les compromis optimaux pour laisser un humain decider.

Le vocabulaire utile pour la suite :

| Terme | Definition |
|-------|------------|
| **Contrainte dure** (*hard constraint*) | Doit etre satisfaite ; si impossible, le probleme est `UNSATISFIABLE`. |
| **Contrainte souple** (*soft constraint*) | Devrait etre satisfaite, mais une violation est toleree moyennant une penalite. |
| **Poids** (*weight*) | Cout numerique attribue a la violation d'une contrainte souple. Plus le poids est eleve, plus Z3 tente de la satisfaire. |
| **Solution dominee** | Il existe une autre solution strictement meilleure sur au moins un objectif, sans etre pire sur aucun autre. |
| **Front de Pareto** | Ensemble des solutions non-dominees : les compromis optimaux. |

## 2. Objectifs hierarchiques avec `Optimize` et priorites

La technique la plus directe pour gerer des contraintes de priorite variable consiste a introduire des **variables de relaxation booleennes**. Pour chaque contrainte souple `c_i`, on cree une variable `r_i` (un `BoolExpr`) et on ajoute `MkOr(c_i, r_i)` : la contrainte peut etre violee, mais seulement si `r_i` vaut `True`. On minimise ensuite la somme ponderee des `r_i`.

### Principe

Soit un ensemble de contraintes souples $c_1, c_2, \ldots, c_k$ avec des poids $w_1, w_2, \ldots, w_k$. Pour chaque $c_i$ :

$$\text{ajouter la contrainte relachee : } c_i \lor r_i$$

puis minimiser le cout total des violations :

$$\text{minimiser } \sum_{i=1}^{k} w_i \cdot \mathbb{1}[r_i = \text{True}]$$

En C#/.NET, `ctx.MkITE(r_i, ctx.MkInt(weight_i), ctx.MkInt(0))` construit exactement l'indicateur $w_i \cdot \mathbb{1}[r_i]$.

### Exemple : ordonnanceur avec contraintes dures et souples

Trois taches doivent etre planifiees dans des fenetres temporelles. Certaines contraintes sont **imperatives** (dures), d'autres sont **souhaitees** (souples) avec des poids differents : une priorite elevee signifie que Z3 fera de son mieux pour la satisfaire.

In [2]:
// Ordonnanceur hierarchique : contraintes dures + contraintes souples ponderees.
// Trois taches T0, T1, T2. Chaque tache i demarre a debut_i (entier >= 0),
// dure 1 unite de temps, et deux taches ne peuvent s'executer au meme instant.
var opt = ctx.MkOptimize();

// Variables de decision : creneau de debut de chaque tache (0 a 9)
var debut = new IntExpr[3];
for (int i = 0; i < 3; i++) {
    debut[i] = ctx.MkIntConst($"debut_{i}");
    opt.Assert(ctx.MkGe(debut[i], ctx.MkInt(0)), ctx.MkLe(debut[i], ctx.MkInt(9)));
}

// --- Contraintes DURES (hard) ---
// Les trois taches doivent occuper des creneaux distincts.
opt.Assert(ctx.MkDistinct(debut));
// T0 doit absolument commencer apres le creneau 2 (contrainte externe)
opt.Assert(ctx.MkGe(debut[0], ctx.MkInt(2)));

// --- Contraintes SOUPLES (soft) avec poids ---
// (description, contrainte, poids)
var preferences = new (string, BoolExpr, int)[] {
    ("T0 le plus tot possible (debut_0 == 2)", ctx.MkEq(debut[0], ctx.MkInt(2)), 10),
    ("T1 avant T0 (debut_1 < debut_0)",         ctx.MkLt(debut[1], debut[0]),     7),
    ("T2 en dernier (debut_2 > debut_0)",        ctx.MkGt(debut[2], debut[0]),     5),
    ("T1 au creneau 0 (debut_1 == 0)",           ctx.MkEq(debut[1], ctx.MkInt(0)), 3),
};

var relaxVars = new List<(string desc, BoolExpr r, int poids)>();
var coutTotal = (ArithExpr)ctx.MkInt(0);
for (int idx = 0; idx < preferences.Length; idx++) {
    var (desc, contrainte, poids) = preferences[idx];
    var r = ctx.MkBoolConst($"r_{idx}");
    relaxVars.Add((desc, r, poids));
    // Contrainte relachee : c_i OU r_i
    opt.Assert(ctx.MkOr(contrainte, r));
    coutTotal = ctx.MkAdd(coutTotal, (ArithExpr)ctx.MkITE(r, ctx.MkInt(poids), ctx.MkInt(0)));
}
var coutVar = ctx.MkIntConst("cout_total");
opt.Assert(ctx.MkEq(coutVar, coutTotal));
opt.MkMinimize(coutVar);

Console.WriteLine($"Ordonnancement hierarchique : {opt.Check()}");
if (opt.Check() == Status.SATISFIABLE) {
    var m = opt.Model;
    Console.WriteLine("\nPlanning obtenu :");
    for (int i = 0; i < 3; i++)
        Console.WriteLine($"  T{i} : creneau {((IntNum)m.Evaluate(debut[i])).Int64}");
    Console.WriteLine("\nContraintes souples :");
    long coutCumul = 0;
    foreach (var (desc, r, poids) in relaxVars) {
        bool violee = ((BoolExpr)m.Evaluate(r)).BoolValue == Z3_lbool.Z3_L_TRUE;
        long penalite = violee ? poids : 0;
        coutCumul += penalite;
        string statut = violee ? "VIOLEE" : "satisfaite";
        Console.WriteLine($"  [{statut,9}] (poids {poids,2}) {desc}");
    }
    Console.WriteLine($"\nCout total des violations : {((IntNum)m.Evaluate(coutVar)).Int64}");
}

Ordonnancement hierarchique : SATISFIABLE



Planning obtenu :


  T0 : creneau 2


  T1 : creneau 0


  T2 : creneau 9


...

  [satisfaite] (poids 10) T0 le plus tot possible (debut_0 == 2)


  [satisfaite] (poids  7) T1 avant T0 (debut_1 < debut_0)


  [satisfaite] (poids  5) T2 en dernier (debut_2 > debut_0)


  [satisfaite] (poids  3) T1 au creneau 0 (debut_1 == 0)



Cout total des violations : 0


### Interpretation : hierarchie ponderee

**Sortie obtenue** : Z3 trouve un planning qui satisfait toutes les contraintes dures et minimise la somme ponderee des violations des contraintes souples (cout total nul ici, car toutes les preferences sont compatibles).

| Mecanisme | API Microsoft.Z3 | Role |
|-----------|------------------|------|
| Variable de relaxation | `ctx.MkBoolConst($"r_{i}")` | Vaut `True` si la contrainte souple est violee |
| Contrainte relachee | `ctx.MkOr(contrainte, r_i)` | Permet la violation via `r_i` |
| Penalite | `ctx.MkITE(r_i, ctx.MkInt(poids), ctx.MkInt(0))` | Contribution au cout total si violation |
| Objectif | `opt.MkMinimize(cout_total)` | Minimiser la somme des penalites |

**Points cles** :
1. Les contraintes a **poids eleve** sont prioritaires : Z3 prefere violer plusieurs contraintes a faible poids qu'une seule a poids eleve.
2. Toutes les contraintes dures sont satisfaites par construction (elles sont ajoutees avec `opt.Assert` sans relaxation).
3. Le cout total obtenu est le **minimum** : aucune autre assignation ne donne une somme ponderee de violations inferieure.

> **Note technique (.NET)** : Microsoft.Z3 expose aussi `opt.AssertSoft(BoolExpr, uint weight, string group)` qui automatise le schema relaxation+penalite pour un groupe de contraintes. On l'illustre en section 5. Le codage manuel ci-dessus reste utile pour comprendre le mecanisme et controler finement l'expression du cout. Le choix des poids est decisive : en pratique, on utilise souvent une echelle exponentielle (1, 2, 4, 8...) pour garantir une veritable hierarchie lexicographique approximative.

## 3. Objectifs multiples -- `MkMaximize` vs `MkMinimize` simultanes

Un `Optimize` peut contenir **plusieurs objectifs** declares via `opt.MkMaximize(...)` et `opt.MkMinimize(...)`. Z3 resout ces objectifs de maniere **lexicographique** (par ordre de declaration) : le premier objectif est optimise en priorite, puis le second est optimise sous la contrainte que le premier reste optimal.

### Lecture des bornes

Chaque appel a `opt.MkMaximize(expr)` ou `opt.MkMinimize(expr)` renvoie un **handle** de type `Optimize.Handle`. Apres `opt.Check()`, on peut interroger :

- `handle.Upper` : borne superieure de l'objectif (utile apres `MkMaximize`).
- `handle.Lower` : borne inferieure de l'objectif (utile apres `MkMinimize`).

Ces proprietes retournent un `Expr` (en pratique un `IntNum` pour les objectifs entiers) : on caste en `IntNum` puis `.Int64` pour la valeur.

### Exemple : maximiser le revenu, puis minimiser le cout

Une entreprise choisit combien d'unites produire (`q`, entier). Chaque unite generee un revenu de 8 EUR mais coute 3 EUR en production. La capacite est limitee a 15 unites. On veut **maximiser le revenu** (priorite 1), puis **minimiser le cout** (priorite 2).

In [3]:
// Optimisation multi-objectif lexicographique.
// Objectif 1 (priorite haute) : maximiser le revenu = 8 * q
// Objectif 2 (priorite basse) : minimiser le cout = 3 * q
var optLex = ctx.MkOptimize();
var q = ctx.MkIntConst("q");
optLex.Assert(ctx.MkGe(q, ctx.MkInt(0)), ctx.MkLe(q, ctx.MkInt(15)));

var revenu = ctx.MkMul(ctx.MkInt(8), q);
var cout = ctx.MkMul(ctx.MkInt(3), q);

// Declaration dans l'ordre de priorite (lexicographique)
var hRevenu = optLex.MkMaximize(revenu);  // priorite 1 : maximiser le revenu
var hCout = optLex.MkMinimize(cout);      // priorite 2 : minimiser le cout

Console.WriteLine($"Multi-objectif lexicographique : {optLex.Check()}");
if (optLex.Check() == Status.SATISFIABLE) {
    var m = optLex.Model;
    long qVal = ((IntNum)m.Evaluate(q)).Int64;
    Console.WriteLine($"\nSolution : q = {qVal} unites");
    Console.WriteLine($"  Revenu = 8 x {qVal} = {8 * qVal} EUR (maximise en priorite)");
    Console.WriteLine($"  Cout   = 3 x {qVal} = {3 * qVal} EUR (minimise ensuite)");
    Console.WriteLine($"  Profit net = {8 * qVal - 3 * qVal} EUR");
    Console.WriteLine($"\nBornes : revenu <= {((IntNum)hRevenu.Upper).Int64}, cout >= {((IntNum)hCout.Lower).Int64}");
}

// Comparaison : maximisation du profit net (5 * q)
var opt2 = ctx.MkOptimize();
var q2 = ctx.MkIntConst("q2");
opt2.Assert(ctx.MkGe(q2, ctx.MkInt(0)), ctx.MkLe(q2, ctx.MkInt(15)));
var profit = ctx.MkSub(ctx.MkMul(ctx.MkInt(8), q2), ctx.MkMul(ctx.MkInt(3), q2));
opt2.MkMaximize(profit);
Console.WriteLine("\n--- Comparaison : maximisation du profit net (5 * q) ---");
Console.WriteLine($"Resultat : {opt2.Check()}");
if (opt2.Check() == Status.SATISFIABLE) {
    var m2 = opt2.Model;
    long q2Val = ((IntNum)m2.Evaluate(q2)).Int64;
    Console.WriteLine($"  q = {q2Val}, profit = {5 * q2Val} EUR");
    Console.WriteLine("\nDans cet exemple les deux approches convergent (profit croissant en q),");
    Console.WriteLine("mais en general lexicographique != somme ponderee.");
}

Multi-objectif lexicographique : SATISFIABLE



Solution : q = 15 unites


  Revenu = 8 x 15 = 120 EUR (maximise en priorite)


  Cout   = 3 x 15 = 45 EUR (minimise ensuite)


  Profit net = 75 EUR


...


--- Comparaison : maximisation du profit net (5 * q) ---


Resultat : SATISFIABLE


  q = 15, profit = 75 EUR



Dans cet exemple les deux approches convergent (profit croissant en q),


mais en general lexicographique != somme ponderee.


### Interpretation : lexicographique vs pondere

**Sortie obtenue** : Z3 trouve `q = 15` (capacite maximale), ce qui maximise le revenu. Le cout est ensuite minimise, mais comme `cout = 3 * q` et que `q` est deja fixe par la maximisation du revenu, le cout ne peut pas etre reduit independamment.

| Strategie | Comment ca marche | Quand l'utiliser |
|-----------|-------------------|------------------|
| **Lexicographique** | Optimise les objectifs dans l'ordre de declaration | Priorites claires et strictes (A domine B) |
| **Ponderee** (section 2) | Minimise la somme ponderee des violations | Compromis souhaites entre objectifs |
| **Pareto** (section 4) | Enumere tous les compromis optimaux | Pas de priorite naturelle, decision humaine |

**Points cles** :
1. `handle.Upper` donne la valeur optimale d'un objectif `MkMaximize` ; `handle.Lower` pour `MkMinimize` (caster le `Expr` retourne en `IntNum`).
2. L'ordre de declaration des `MkMaximize`/`MkMinimize` definit la **priorite lexicographique**.
3. L'approche lexicographique n'est **pas equivalente** a maximiser une somme ponderee : elle impose une hierarchie stricte.

> **Note technique** : L'optimisation multi-objectif lexicographique de Z3 suit le schema Box/Wilson : optimiser l'objectif 1, fixer sa valeur optimale comme contrainte, puis optimiser l'objectif 2, et ainsi de suite.

## 4. Front de Pareto -- explorer les compromis

Quand deux objectifs sont contradictoires et qu'aucune priorite naturelle n'existe, la notion de front de Pareto est pertinente. Une solution est dite **Pareto-optimale** si aucune autre solution ne l'ameliore sur un objectif sans la degrader sur l'autre. Le front de Pareto est l'ensemble de toutes ces solutions optimales.

### Methode d'enumeration

Pour construire le front de Pareto entre maximiser $A$ et minimiser $B$ :

1. Maximiser $A$ et enregistrer $(A^*, B^*)$.
2. Ajouter la contrainte $B < B^*$ (imposer un cout strictement inferieur).
3. Re-maximiser $A$ sous cette nouvelle contrainte.
4. Repeter jusqu'a `UNSATISFIABLE`.

On obtient ainsi une suite de points $(A_1, B_1), (A_2, B_2), \ldots$ ou le cout decroit et la qualite s'ajuste.

### Exemple : qualite vs cout

On choisit un niveau de qualite `q` (0 a 10) et un niveau de cout `c`. Les deux sont lies : une qualite elevee implique un cout minimal, mais le cout peut aussi augmenter pour d'autres raisons. On cherche tous les compromis optimaux (qualite maximale, cout minimal).

In [4]:
// Enumeration du front de Pareto : maximiser qualite (0-10), minimiser cout.
// Relation : une qualite q exige un cout minimal de q * 2.
List<(long qualite, long cout)> EnumererFrontPareto(Context ctx, int maxIter = 15) {
    var frontBrut = new List<(long, long)>();
    for (int iteration = 0; iteration < maxIter; iteration++) {
        var optL = ctx.MkOptimize();
        var qualite = ctx.MkIntConst("qualite");
        var cout = ctx.MkIntConst("cout");
        // Domaine : petits entiers pour garantir un calcul rapide
        optL.Assert(ctx.MkGe(qualite, ctx.MkInt(0)), ctx.MkLe(qualite, ctx.MkInt(10)));
        optL.Assert(ctx.MkGe(cout, ctx.MkInt(0)), ctx.MkLe(cout, ctx.MkInt(20)));
        // Relation qualite-cout : une qualite q exige un cout minimal de q * 2
        optL.Assert(ctx.MkGe(cout, ctx.MkMul(ctx.MkInt(2), qualite)));
        // Exclusion : forcer un cout strictement plus bas que le precedent
        foreach (var (_, cPrec) in frontBrut)
            optL.Assert(ctx.MkLt(cout, ctx.MkInt(cPrec)));
        // Objectif : maximiser la qualite
        optL.MkMaximize(qualite);
        if (optL.Check() != Status.SATISFIABLE) break;  // plus de solution : front complet
        var m = optL.Model;
        long qVal = ((IntNum)m.Evaluate(qualite)).Int64;
        long cVal = ((IntNum)m.Evaluate(cout)).Int64;
        frontBrut.Add((qVal, cVal));
    }
    // Dedoublonner : ne garder qu'un point par niveau de qualite
    var front = new List<(long, long)>();
    var vues = new HashSet<long>();
    foreach (var (q, c) in frontBrut) {
        if (vues.Add(q)) front.Add((q, c));
    }
    return front;
}

var front = EnumererFrontPareto(ctx);
Console.WriteLine("Front de Pareto (qualite max, cout min) :");
Console.WriteLine($"{"Point",6} | {"Qualite",7} | {"Cout",5} | {"Cout min = 2*q",15}");
Console.WriteLine(new string('-', 45));
for (int i = 0; i < front.Count; i++) {
    var (q, c) = front[i];
    long coutMin = q * 2;
    string marque = c == coutMin ? " <-- cout min" : "";
    Console.WriteLine($"{i + 1,6} | {q,7} | {c,5} | {coutMin,15}{marque}");
}
Console.WriteLine($"\n{front.Count} points Pareto-optimaux trouves (qualites distinctes).");
Console.WriteLine("Chaque point est un compromis : pour baisser le cout, il faut");
Console.WriteLine("sacrifier de la qualite (puisque cout_min = 2 * qualite).");

Front de Pareto (qualite max, cout min) :


 Point | Qualite |  Cout |  Cout min = 2*q


---------------------------------------------


     1 |      10 |    20 |              20 <-- cout min


     2 |       9 |    19 |              18


...

     9 |       2 |     5 |               4


    10 |       1 |     2 |               2 <-- cout min



10 points Pareto-optimaux trouves (qualites distinctes).


Chaque point est un compromis : pour baisser le cout, il faut


sacrifier de la qualite (puisque cout_min = 2 * qualite).


### Visualiser le front de Pareto

Un tableau de nombres ne rend pas justice au concept de **compromis**. En projetant chaque point Pareto-optimal dans le plan (qualite, cout), le front devient une courbe descendante : chaque pas vers un cout plus bas coute de la qualite. La droite `cout_min = 2 * qualite` (la frontiere de faisabilite imposee par le modele) apparait en pointille : on voit immediatement que certains points Pareto-optimaux sont *strictement au-dessus* de cette borne minimale -- Z3 les a choisis parce qu'a qualite fixee, plusieurs couts sont Pareto-equivalents, et le solveur en echantillonne un. C'est precisement ce que le front de Pareto revele qu'un simple « minimiser le cout » cacherait.

> Le twin Python trace cette courbe avec **matplotlib** (rendu PNG). Le twin C# utilise un rendu **ASCII** (matrice de caracteres) pour rester autonome, sans dependance graphique.

In [5]:
// Visualisation ASCII du front de Pareto : qualite (x) vs cout (y).
// 'O' = points Pareto-optimaux ; '.' = droite cout_min = 2*qualite.
void AfficherFrontPareto(List<(long q, long c)> front) {
    int W = 34, H = 16;            // largeur / hauteur (caracteres)
    long qMax = 10, cMax = 20;
    var grille = new char[H, W];
    for (int y = 0; y < H; y++)
        for (int x = 0; x < W; x++) grille[y, x] = ' ';
    // Front de Pareto (O)
    foreach (var (q, c) in front) {
        int x = (int)(q * (W - 1) / qMax);
        int y = (int)(c * (H - 1) / cMax);   // y=0 en haut = cout max
        if (y >= 0 && y < H && x >= 0 && x < W) grille[y, x] = 'O';
    }
    // Borne cout_min = 2*qualite (.)
    for (long q = 0; q <= qMax; q++) {
        long c = 2 * q;
        if (c < 0 || c > cMax) continue;
        int x = (int)(q * (W - 1) / qMax);
        int y = (int)(c * (H - 1) / cMax);
        if (y >= 0 && y < H && x >= 0 && x < W && grille[y, x] == ' ') grille[y, x] = '.';
    }
    Console.WriteLine("Front de Pareto (O) vs cout_min = 2*qualite (.)");
    Console.WriteLine($"cout {cMax,3} +");
    for (int y = 0; y < H; y++) {
        Console.Write("       |");
        for (int x = 0; x < W; x++) Console.Write(grille[y, x]);
        Console.WriteLine();
    }
    Console.WriteLine($"cout {0,3} +{new string('-', W)}  qualite -> {qMax}");
}
AfficherFrontPareto(front);
Console.WriteLine($"\nLe front comporte {front.Count} compromis Pareto-optimaux.");
Console.WriteLine("La pente descendante illustre le trade-off : baisser le cout degrade la qualite.");

Front de Pareto (O) vs cout_min = 2*qualite (.)


cout  20 +


       |

.

       |

...

       |

O

cout   0 +----------------------------------  qualite -> 10



Le front comporte 10 compromis Pareto-optimaux.


La pente descendante illustre le trade-off : baisser le cout degrade la qualite.


### Interpretation : front de Pareto

**Sortie obtenue** : une liste de points `(qualite, cout)` tries par cout decroissant. Chaque point est Pareto-optimal : on ne peut pas ameliorer un objectif sans degrader l'autre.

| Etape | Action | Resultat |
|-------|--------|----------|
| 1 | `opt.MkMaximize(qualite)` sans contrainte de cout | Point avec qualite maximale |
| 2 | Ajouter `cout < cout_precedent`, re-maximiser | Point avec cout plus bas |
| 3 | Repeter jusqu'a `UNSATISFIABLE` | Front complet |

**Points cles** :
1. Le front de Pareto contient les **compromis optimaux** : aucun point n'est domine par un autre.
2. Le nombre de points est fini (domains entiers petits), mais peut etre grand si les ranges sont larges.
3. Cette methode d'enumeration par exclusion successive est simple mais couteuse : a chaque iteration, on ajoute une contrainte. Pour de grands fronts, on prefere des algorithmes specialises.

> **Note technique** : On maintient les domains entiers **petits** (0-10 pour la qualite, 0-20 pour le cout) pour garantir que chaque appel a `opt.Check()` est quasi instantane. Avec de larges ranges de `Real`, l'enumeration du front de Pareto peut devenir prohibitive.

## Exercice 1 : Construire un front de Pareto

### Enonce

Ecrivez une fonction `ConstruireFrontPareto` qui enumere le front de Pareto pour maximiser une variable `qualite` (entier 0-10) et minimiser une variable `cout` (entier 0-20), lies par la contrainte `cout >= qualite * 2`.

La fonction doit retourner une liste de couples `(qualite, cout)` representant tous les compromis optimaux.

### Indices

- `# Indice` : a chaque iteration, creez un nouvel `Optimize`, maximisez `qualite`, puis ajoutez `cout < cout_dernier_point` comme contrainte pour la prochaine iteration.
- `# Etape 1` : initialiser `front = new List<(long,long)>()` et boucler (`for (int it = 0; it < maxIter; it++)`).
- `# Etape 2` : dans la boucle, creer `ctx.MkOptimize()`, declarer `qualite` et `cout` avec leurs bornes.
- `# Etape 3` : ajouter `cout >= qualite * 2` et, pour chaque point precedent, `cout < cout_prec`.
- `# Etape 4` : `opt.MkMaximize(qualite)` puis `opt.Check()` ; si `UNSATISFIABLE`, `break`.
- `# Etape 5` : extraire les valeurs et les ajouter a `front`.

In [6]:
// EXERCICE 1 : Enumerer le front de Pareto (qualite vs cout).
List<(long qualite, long cout)> ConstruireFrontPareto(Context ctx, int maxIter = 15) {
    Console.WriteLine("Exercice 1 - a completer");
    // TODO etudiant : implementez l'enumeration du front de Pareto
    // Indice : bouclez, a chaque iteration maximisez qualite, enregistrez
    //          (qualite, cout), puis ajoutez cout < cout_actuel comme contrainte.
    // Etape 1 : initialiser front = new List<(long,long)>()
    // Etape 2 : boucler (for iteration = 0; iteration < maxIter; iteration++)
    // Etape 3 : creer ctx.MkOptimize(), ajouter les bornes et cout >= qualite * 2
    // Etape 4 : ajouter cout < cout_prec pour chaque point deja trouve
    // Etape 5 : opt.MkMaximize(qualite), si UNSATISFIABLE -> break, sinon extraire et ajouter
    return null;  // TODO etudiant : remplacer par la liste des couples (qualite, cout)
}

var frontEx1 = ConstruireFrontPareto(ctx);
Console.WriteLine($"Front de Pareto : {(frontEx1 == null ? "(a completer)" : string.Join(", ", frontEx1))}");

Exercice 1 - a completer


Front de Pareto : (a completer)


## 5. Contraintes souples (soft constraints) et MaxSAT

Le probleme **MaxSAT** consiste a satisfaire un **maximum** de contraintes souples quand toutes ne peuvent l'etre simultanement. C'est un cas particulier de l'approche ponderee de la section 2, ou toutes les contraintes ont le meme poids (on compte simplement le nombre de violations).

### Formalisation

Soit $k$ contraintes souples $c_1, \ldots, c_k$. Pour chacune, on introduit une variable de relaxation $r_i \in \{0, 1\}$ et on ajoute $c_i \lor r_i$. On minimise ensuite :

$$\sum_{i=1}^{k} r_i$$

La valeur optimale donne le **nombre minimum de contraintes violees**.

### Exemple : assignation de salles avec preferences

Quatre etudiants doivent etre assignes a quatre salles (une chacun). Chaque etudiant a des preferences (salle preferee). Toutes les preferences ne sont pas compatibles (deux etudiants peuvent preferer la meme salle). On veut satisfaire un **maximum** de preferences.

In [7]:
// MaxSAT : satisfaire un maximum de preferences d'assignation de salles.
// 4 etudiants (E0..E3), 4 salles (S0..S3). Assignation bijective.
// Preferences (potentiellement conflictuelles) :
//   E0 prefere S0, E1 prefere S0 (conflit !), E2 prefere S2, E3 prefere S1.
var optMax = ctx.MkOptimize();
int n = 4;
// Variable : salle[i] = numero de salle assignee a l'etudiant i
var salle = new IntExpr[n];
for (int i = 0; i < n; i++) {
    salle[i] = ctx.MkIntConst($"salle_{i}");
    optMax.Assert(ctx.MkGe(salle[i], ctx.MkInt(0)), ctx.MkLe(salle[i], ctx.MkInt(3)));
}
// Contrainte DURE : assignation bijective (chaque salle a exactement un etudiant)
optMax.Assert(ctx.MkDistinct(salle));

// Preferences (contraintes SOUPLES, toutes de poids 1 = MaxSAT uniforme)
var preferences = new (int etudiant, int sallePref)[] { (0, 0), (1, 0), (2, 2), (3, 1) };
var relaxVars = new List<(int etudiant, int sallePref, BoolExpr r)>();
var sommeViolations = (ArithExpr)ctx.MkInt(0);
foreach (var (etudiant, sallePref) in preferences) {
    var r = ctx.MkBoolConst($"pref_{etudiant}_{sallePref}");
    relaxVars.Add((etudiant, sallePref, r));
    // Contrainte relachee : etudiant obtient sa salle OU r est vrai
    optMax.Assert(ctx.MkOr(ctx.MkEq(salle[etudiant], ctx.MkInt(sallePref)), r));
    sommeViolations = ctx.MkAdd(sommeViolations, (ArithExpr)ctx.MkITE(r, ctx.MkInt(1), ctx.MkInt(0)));
}
// Objectif : minimiser le nombre de preferences non satisfaites
var nbViolations = ctx.MkIntConst("nb_violations");
optMax.Assert(ctx.MkEq(nbViolations, sommeViolations));
optMax.MkMinimize(nbViolations);

Console.WriteLine($"MaxSAT (assignation de salles) : {optMax.Check()}");
if (optMax.Check() == Status.SATISFIABLE) {
    var m = optMax.Model;
    Console.WriteLine("\nAssignation optimale :");
    int nbSatisfaites = 0;
    foreach (var (etudiant, sallePref, r) in relaxVars) {
        long s = ((IntNum)m.Evaluate(salle[etudiant])).Int64;
        bool violee = ((BoolExpr)m.Evaluate(r)).BoolValue == Z3_lbool.Z3_L_TRUE;
        if (!violee) nbSatisfaites++;
        string symbole = violee ? "--" : "OK";
        Console.WriteLine($"  E{etudiant} -> S{s}  (preferait S{sallePref}) [{symbole}]");
    }
    Console.WriteLine($"\nPreferences satisfaites : {nbSatisfaites} / {preferences.Length}");
    Console.WriteLine($"Violations minimales : {((IntNum)m.Evaluate(nbViolations)).Int64}");
    Console.WriteLine("\nZ3 ne pouvait pas satisfaire E0 et E1 simultanement (meme preference S0).");
    Console.WriteLine("Il a choisi d'en satisfaire un et sacrifie l'autre (1 violation minimum).");
}

// Variante .NET idiomatic : opt.AssertSoft(c, weight, group) automatise le schema.
// Exemple (commente) equivalent pour E0 -> S0 :
//   optMax.AssertSoft(ctx.MkEq(salle[0], ctx.MkInt(0)), 1u, "prefs");

MaxSAT (assignation de salles) : SATISFIABLE



Assignation optimale :


  E0 -> S3  (preferait S0) [--]


  E1 -> S0  (preferait S0) [OK]


  E2 -> S2  (preferait S2) [OK]


  E3 -> S1  (preferait S1) [OK]



Preferences satisfaites : 3 / 4


Violations minimales : 1



Z3 ne pouvait pas satisfaire E0 et E1 simultanement (meme preference S0).


Il a choisi d'en satisfaire un et sacrifie l'autre (1 violation minimum).


### Interpretation : MaxSAT et variables de relaxation

**Sortie obtenue** : Z3 trouve une assignation qui satisfait 3 des 4 preferences. La seule violation est inevitable car E0 et E1 preferent la meme salle S0.

| Element | Implementation | Role |
|---------|----------------|------|
| Contrainte dure | `ctx.MkDistinct(salle)` | Une salle par etudiant, pas de doublon |
| Contrainte souple | `ctx.MkOr(salle[i] == pref, r_i)` | Preference violable si `r_i = True` |
| Compteur | `ctx.MkITE(r_i, 1, 0)` | Contribution unitaire au nombre de violations |
| Objectif | `opt.MkMinimize(nb_violations)` | Maximiser le nombre de preferences satisfaites |

**Points cles** :
1. Le probleme MaxSAT est un cas particulier d'optimisation ponderee ou tous les poids sont egaux a 1.
2. Les variables de relaxation `Bool` sont l'outil central : elles « absorbent » les violations impossibles a eviter.
3. `ctx.MkDistinct` impose que toutes les valeurs soient distinctes (equivalent AllDifferent).
4. Si on avait donne des **poids differents** aux preferences, Z3 aurait privilege les preferences a poids eleve (retour a la section 2).

> **Note technique (.NET)** : `opt.AssertSoft(BoolExpr c, uint weight, string group)` est la forme native du MaxSAT pondere dans Microsoft.Z3 : les contraintes d'un meme `group` s'agregent, et Z3 minimise la somme ponderee des violations. Le codage manuel ci-dessus reste transparent ; `AssertSoft` est le raccourci idiomatique. MaxSAT est un domaine de recherche actif : Z3 utilise un algorithme de recherche dichotomique sur le nombre de violations pour converger rapidement vers l'optimum.

## Exercice 2 : Satisfaire des preferences (MaxSAT)

### Enonce

Ecrivez une fonction `SatisfairePreferences` qui assigne `nItems` items a `nSlots` creneaux (assignation injective : au plus un item par creneau) de facon a **maximiser** le nombre de preferences satisfaites.

Parametres :
- `nItems` : nombre d'items (entiers)
- `nSlots` : nombre de creneaux disponibles
- `preferences` : liste de couples `(item, slot_prefere)`

Retour attendu : un couple `(assignation, nb_satisfaites)` ou `assignation` est une chaine decrivant l'assignation.

### Indices

- `# Indice` : pour chaque preference, creez une variable de relaxation `Bool` et ajoutez `ctx.MkOr(slot[item] == slot_pref, r)`.
- `# Etape 1` : declarer `slot = new IntExpr[nItems]` avec bornes `[0, nSlots-1]`.
- `# Etape 2` : contrainte dure d'injectivite : `ctx.MkDistinct(slot)` (ou equivalent si `nItems < nSlots`).
- `# Etape 3` : pour chaque `(item, pref)`, creer `r = ctx.MkBoolConst(...)`, ajouter `ctx.MkOr(ctx.MkEq(slot[item], ctx.MkInt(pref)), r)`.
- `# Etape 4` : minimiser la somme des `ctx.MkITE(r, 1, 0)`.
- `# Etape 5` : extraire l'assignation et le nombre de preferences satisfaites.

In [8]:
// EXERCICE 2 : MaxSAT - assignation d'items maximisant les preferences.
(string assignation, int nbSatisfaites) SatisfairePreferences(Context ctx, int nItems, int nSlots, List<(int item, int slot)> preferences) {
    Console.WriteLine("Exercice 2 - a completer");
    // TODO etudiant : implementez la resolution MaxSAT
    // Indice : utilisez des variables Bool de relaxation pour chaque preference.
    // Etape 1 : declarer slot[i] = Int, bornes [0, nSlots-1]
    // Etape 2 : contrainte ctx.MkDistinct(slot) pour l'injectivite
    // Etape 3 : pour chaque (item, pref), ctx.MkOr(ctx.MkEq(slot[item], ctx.MkInt(pref)), r_i)
    // Etape 4 : minimiser la somme des ctx.MkITE(r_i, 1, 0)
    // Etape 5 : extraire assignation et compter les preferences satisfaites
    return ("(a completer)", 0);  // TODO etudiant : remplacer par le resultat
}

var prefsTest = new List<(int, int)> { (0, 0), (1, 0), (2, 1), (3, 1) };
var resultat = SatisfairePreferences(ctx, 4, 4, prefsTest);
Console.WriteLine($"Resultat MaxSAT : assignation = {resultat.assignation}, nb_satisfaites = {resultat.nbSatisfaites}");

Exercice 2 - a completer


Resultat MaxSAT : assignation = (a completer), nb_satisfaites = 0


## 6. Cas pratique -- allocation de budget

Synthesisons les techniques vues (contraintes dures, contraintes souples ponderees, optimisation multi-objectif) sur un cas concret d'**allocation de budget**.

### Scenario

Une organisation dispose d'un budget de **100 unites** a repartir entre 5 projets. Chaque projet $i$ a :
- une **valeur unitaire** $v_i$ (gain par unite investie)
- un **financement minimum** $m_i$ (en-dessous duquel le projet n'est pas viable)
- une **priorite** $p_i$ (1 = haute, 2 = moyenne, 3 = basse)

Objectifs :
1. **(Dur)** Respecter le budget total et les minimums de financement.
2. **(Souple, pondere)** Preferer financer les projets haute priorite au-dela de leur minimum.
3. **(Principal)** Maximiser la valeur totale.

In [9]:
// Allocation de budget multi-projets avec contraintes dures, souples et objectif principal.
var optB = ctx.MkOptimize();
// Donnees : 5 projets (nom, valeur unitaire, financement minimum, priorite)
var projets = new (string nom, int val, int finMin, int prio)[] {
    ("Alpha", 5, 10, 1),   // haute priorite
    ("Beta",  3,  8, 2),   // moyenne priorite
    ("Gamma", 7,  5, 1),   // haute priorite, tres rentable
    ("Delta", 2, 12, 3),   // basse priorite
    ("Epsil", 4,  6, 2),   // moyenne priorite
};
long budgetTotal = 100;
int nProjets = projets.Length;
// Variables : allocation[i] = montant investi dans le projet i
var alloc = new IntExpr[nProjets];
// --- Contraintes DURES ---
for (int i = 0; i < nProjets; i++) {
    alloc[i] = ctx.MkIntConst($"alloc_{projets[i].nom}");
    optB.Assert(ctx.MkGe(alloc[i], ctx.MkInt(projets[i].finMin)));  // minimum viable
    optB.Assert(ctx.MkLe(alloc[i], ctx.MkInt(50)));                 // plafond par projet
}
// Budget total respecte
optB.Assert(ctx.MkLe(ctx.MkAdd(alloc), ctx.MkInt(budgetTotal)));
// --- Contraintes SOUPLES (ponderees) ---
// On souhaite que les projets recoivent au moins 20 unites. Poids inverse de la priorite.
var poidsParPrio = new Dictionary<int, int> { {1, 10}, {2, 5}, {3, 1} };
var coutSouple = (ArithExpr)ctx.MkInt(0);
for (int i = 0; i < nProjets; i++) {
    int poids = poidsParPrio[projets[i].prio];
    var r = ctx.MkBoolConst($"bonus_{projets[i].nom}");
    // Contrainte souple : alloc[i] >= 20 OU penalite
    optB.Assert(ctx.MkOr(ctx.MkGe(alloc[i], ctx.MkInt(20)), r));
    coutSouple = ctx.MkAdd(coutSouple, (ArithExpr)ctx.MkITE(r, ctx.MkInt(poids), ctx.MkInt(0)));
}
var coutViolations = ctx.MkIntConst("cout_violations");
optB.Assert(ctx.MkEq(coutViolations, coutSouple));
// --- Objectif PRINCIPAL : maximiser la valeur totale ---
var valeurExprs = new ArithExpr[nProjets];
for (int i = 0; i < nProjets; i++)
    valeurExprs[i] = ctx.MkMul(ctx.MkInt(projets[i].val), alloc[i]);
var valeurTotale = ctx.MkIntConst("valeur_totale");
optB.Assert(ctx.MkEq(valeurTotale, ctx.MkAdd(valeurExprs)));
// Optimisation lexicographique :
// Priorite 1 : minimiser les violations de contraintes souples (hierarchie)
// Priorite 2 : maximiser la valeur totale
optB.MkMinimize(coutViolations);
optB.MkMaximize(valeurTotale);

Console.WriteLine($"Allocation de budget : {optB.Check()}");
if (optB.Check() == Status.SATISFIABLE) {
    var m = optB.Model;
    Console.WriteLine($"\nBudget total disponible : {budgetTotal}");
    Console.WriteLine($"{"Projet",8} | {"Val/u",5} | {"Min",4} | {"Prio",4} | {"Alloue",6} | {"Valeur",7}");
    Console.WriteLine(new string('-', 50));
    long totalAlloue = 0, totalValeur = 0;
    for (int i = 0; i < nProjets; i++) {
        long a = ((IntNum)m.Evaluate(alloc[i])).Int64;
        long v = projets[i].val * a;
        totalAlloue += a; totalValeur += v;
        Console.WriteLine($"{projets[i].nom,8} | {projets[i].val,5} | {projets[i].finMin,4} | {projets[i].prio,4} | {a,6} | {v,7}");
    }
    Console.WriteLine(new string('-', 50));
    Console.WriteLine($"{"TOTAL",8} | {"",5} | {"",4} | {"",4} | {totalAlloue,6} | {totalValeur,7}");
    Console.WriteLine($"\nCout des violations souples : {((IntNum)m.Evaluate(coutViolations)).Int64}");
    Console.WriteLine($"Budget non utilise : {budgetTotal - totalAlloue}");
}

Allocation de budget : SATISFIABLE



Budget total disponible : 100


  Projet | Val/u |  Min | Prio | Alloue |  Valeur


--------------------------------------------------


   Alpha |     5 |   10 |    1 |     20 |     100


...

   Epsil |     4 |    6 |    2 |     20 |      80


--------------------------------------------------


   TOTAL |       |      |      |    100 |     420



Cout des violations souples : 0


Budget non utilise : 0


### Interpretation : allocation multi-objectif complete

**Sortie obtenue** : Z3 produit une allocation qui respecte tous les minimums de financement (contraintes dures), minimise les violations de bonus prioritaire (contraintes souples), puis maximise la valeur totale. Avec 5 projets et un budget de 100, la solution minimisant les violations donne 20 unites a chaque projet (cout de violations nul).

| Type de contrainte | Mecanisme Microsoft.Z3 | Effet sur la solution |
|--------------------|------------------------|----------------------|
| Budget total | `ctx.MkAdd(alloc) <= 100` | Contrainte dure absolue |
| Minimum par projet | `alloc[i] >= fin_min` | Chaque projet viable |
| Bonus priorite | `ctx.MkOr(alloc[i] >= 20, r)` + poids | Projets prioritaires favorises |
| Valeur totale | `opt.MkMaximize(valeur_totale)` | Optimisee en second lieu |

**Points cles** :
1. Les contraintes dures garantissent la **faisabilite** (budget, minimums).
2. Les contraintes souples hierarchisent les **preferences** (favoriser les projets prioritaires).
3. L'objectif principal maximise la **valeur** une fois les preferences honorees.
4. L'ordre `MkMinimize` puis `MkMaximize` impose la lexicographie : les violations sont eliminees en priorite, puis la valeur est maximisee.

> **Note technique** : Ce cas pratique combine les trois techniques du notebook : relaxation booleenne (section 2/5), lexicographie (section 3) et contraintes dures (NB01). C'est le pattern typique des problemes d'allocation de ressources reels.

## Exercice 3 : Allocation de budget avec priorites

### Enonce

Ecrivez une fonction `AllouerBudget` qui alloue un budget fixe entre plusieurs projets pour maximiser la valeur totale, tout en respectant des contraintes de financement minimum et de priorite.

Parametres :
- `budgetTotal` : entier, budget disponible
- `projets` : liste de tuples `(nom, valeur_unitaire, financement_min, priorite)` ou priorite 1 = haute, 2 = moyenne, 3 = basse

Retour attendu : un couple `(allocations, valeur_totale)` ou `allocations` est une chaine decrivant les montants.

### Indices

- `# Indice` : utilisez `ctx.MkOptimize()` avec `opt.MkMaximize(valeur_totale)` et des contraintes de minimum.
- `# Etape 1` : declarer une variable `Int` par projet avec bornes `[financement_min, budget_total]`.
- `# Etape 2` : contrainte dure `ctx.MkAdd(allocations) <= budget_total`.
- `# Etape 3` : (optionnel) contraintes souples : `ctx.MkOr(alloc[i] >= seuil_priorite, r_i)` avec poids selon la priorite.
- `# Etape 4` : `valeur_totale = ctx.MkAdd(valeur_unitaire[i] * alloc[i])`, puis `opt.MkMaximize(valeur_totale)`.
- `# Etape 5` : extraire les allocations et calculer la valeur totale.

In [10]:
// EXERCICE 3 : Allocation de budget avec priorites.
(string allocations, long valeurTotale) AllouerBudget(Context ctx, long budgetTotal, List<(string nom, int val, int finMin, int prio)> projets) {
    Console.WriteLine("Exercice 3 - a completer");
    // TODO etudiant : implementez l'allocation optimale
    // Indice : MkOptimize + MkMaximize(valeur) + contraintes de minimum.
    // Etape 1 : declarer alloc par projet, bornee par [finMin, budgetTotal]
    // Etape 2 : contrainte ctx.MkAdd(allocs) <= budgetTotal
    // Etape 3 : (optionnel) contraintes souples selon la priorite
    // Etape 4 : valeur = ctx.MkAdd(val_unit * alloc), opt.MkMaximize(valeur)
    // Etape 5 : extraire allocations et valeur totale
    return ("(a completer)", 0);  // TODO etudiant : remplacer par le resultat
}

var projetsTest = new List<(string, int, int, int)> {
    ("Alpha", 5, 10, 1), ("Beta", 3, 8, 2), ("Gamma", 7, 5, 1), ("Delta", 2, 12, 3),
};
var resultatBudget = AllouerBudget(ctx, 80, projetsTest);
Console.WriteLine($"Allocation optimale : {resultatBudget.allocations}, valeur = {resultatBudget.valeurTotale}");

Exercice 3 - a completer


Allocation optimale : (a completer), valeur = 0


## Recapitulatif

Ce notebook a explore les techniques d'optimisation avancee de Z3 au-dela du simple `MkMaximize`/`MkMinimize` d'un seul objectif :

| Technique | API Microsoft.Z3 | Quand l'utiliser |
|-----------|------------------|------------------|
| **Priorites hierarchiques** | relaxation `Bool` + `MkITE(r, poids, 0)` + `MkMinimize` | Contraintes souples avec importance differente |
| **Objectifs multiples lexicographiques** | `opt.MkMaximize` puis `opt.MkMinimize` (ordre de declaration) | Priorites strictes entre objectifs (A domine B) |
| **Front de Pareto** | Boucle `MkMaximize` + exclusion successive | Explorer tous les compromis optimaux |
| **MaxSAT uniforme** | relaxation `Bool` + `MkMinimize(Sum(MkITE(r,1,0)))` -- ou `AssertSoft` | Satisfaire un maximum de preferences equiponderees |

**Points essentiels a retenir** :

1. **Variables de relaxation `Bool`** : c'est l'outil universel pour les contraintes souples. Une contrainte `ctx.MkOr(c, r)` peut etre violee si `r = True`, et on penalise cette violation dans l'objectif.
2. **Lexicographique vs pondere** : l'optimisation lexicographique (ordre des `MkMaximize`/`MkMinimize`) impose une hierarchie stricte ; la somme ponderee permet des compromis nuances.
3. **Front de Pareto** : indispensable quand aucun ordre naturel n'existe entre objectifs. On l'enumere par exclusion successive (forcer l'autre objectif a s'ameliorer a chaque tour).
4. **Domains petits** : pour l'enumeration du front de Pareto et les boucles d'optimisation, garder des domains entiers petits (0-20) garantit des temps de calcul raisonnables.
5. **Pattern d'allocation** : le cas pratique (section 6) combine tous ces outils : contraintes dures (budget), contraintes souples (priorites), objectif principal (valeur). C'est le squelette de la plupart des problemes d'allocation de ressources reels.

Ces techniques font de Z3 un outil puissant non seulement pour la **satisfaction** de contraintes, mais aussi pour l'**optimisation multi-criteres** -- un domaine ou la modelisation declarative brille face aux approches ad-hoc.

---

**Parite .NET** : ce twin C# reproduit fidelement le notebook Python [Z3-Python-06-Advanced-Optimization.ipynb](Z3-Python-06-Advanced-Optimization.ipynb) avec le **meme moteur** C++ sous-jacent (Microsoft.Z3 / z3-solver). Les differences sont purement d'API : `Optimize()` -> `ctx.MkOptimize()`, `opt.add()` -> `opt.Assert()`, `opt.maximize/minimize` -> `opt.MkMaximize/MkMinimize` (retournent un `Handle` dont `.Upper`/`.Lower` donnent les bornes), et la visualisation matplotlib est remplacee par un rendu ASCII autonome. La serie Z3-Python-Csharp est maintenant complete (6/6).